In [ ]:
!pip install uv

In [1]:
!uv pip install openai pandas openpyxl arize arize-otel openinference-instrumentation-openai openai-agents

Audited 7 packages in 36ms


In [ ]:
!git clone https://github.com/rk-yen/customer-support-collab.git

# Secrets (Before Starting)
Add the following variables in the Secrets section:
- `ARIZE_API_KEY`
- `ARIZE_SPACE_ID`
- `OPENAI_API_KEY`

In [2]:
from pathlib import Path
import json
import sys

import pandas as pd
pd.set_option('max_colwidth', None)

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / 'workshop_helpers').exists():
    candidates = [path for path in REPO_ROOT.iterdir() if path.is_dir() and (path / 'workshop_helpers').exists()]
    if not candidates:
        raise FileNotFoundError('Could not find a repo folder containing workshop_helpers')
    REPO_ROOT = candidates[0]

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from workshop_helpers.backend import TOOLS, hydrate_backend_from_dataset, snapshot_backend
from workshop_helpers.data import DATASET
from workshop_helpers.experiments import (
    VARIANT_BEHAVIORS,
    build_review_context_message,
    dispatch_specialist_response,
    ensure_arize_dataset,
    format_checklist_rows,
    prepare_experiment_bundle,
    production_readiness_checklist,
    run_experiment,
    run_router_local,
    select_cases,
    select_cases_by_category,
    summarize_dataset,
    summarize_router_experiment_results,
)
from workshop_helpers.metrics import (
    ExactMatchEvaluator,
    escalation_decision_result,
    judge_brand_voice,
    judge_helpfulness,
    pack_response_payload,
    score_routing_response,
)
from workshop_helpers.scenarios import run_router_structured

In [3]:
from workshop_helpers.setup import setup_clients

env = setup_clients()
client = env['client']
ARIZE_SPACE_ID = env['arize_space_id']
ARIZE_API_KEY = env['arize_api_key']

🔭 OpenTelemetry Tracing Details 🔭
|  Arize Project: Workiva-ProblemFirst-Workshop-2026-04-07
|  Span Processor: BatchSpanProcessor
|  Collector Endpoint: otlp.arize.com
|  Transport: gRPC
|  Transport Headers: {'authorization': '****', 'api_key': '****', 'arize-space-id': '****', 'space_id': '****', 'arize-interface': '****'}
|  
|  Using a default SpanProcessor. `add_span_processor` will overwrite this default.
|  
|  `register` has set this TracerProvider as the global OpenTelemetry default.
|  To disable this behavior, call `register` with `set_global_tracer_provider=False`.



# Workiva - AI Support Copilot Workshop

We will work through two systems:
1. `v1` router classification with a code-based exact-match metric.
2. `v2` routed draft-reply creation with three specialist behaviors:
   - `permissions`: single LLM call
   - `review_workflow`: LLM call with injected workflow context
   - `billing`: tool-using billing agent

The routing approach is deterministic structured output routing.
The router selects a support category, then the notebook dispatches to the matching specialist.

## Setup

The scenario library has 50 cases total.
Use `LIMIT_N_CASES` for a faster workshop run if needed.

In [4]:
LIMIT_N_CASES = 6  # set to an integer like 12 for a faster run
EXPERIMENT_DATASET = select_cases(DATASET, limit_n=LIMIT_N_CASES)
library_summary = summarize_dataset(DATASET)
experiment_summary = summarize_dataset(EXPERIMENT_DATASET)
backend_snapshot = hydrate_backend_from_dataset(DATASET)

print(f"Scenario library: {library_summary['scenario_count']} total cases")
print(f"Experiment dataset: {experiment_summary['scenario_count']} cases")
print(f"Categories: {', '.join(experiment_summary['categories'])}")
print(f"Limit applied: {LIMIT_N_CASES}")
print()
print('Demo backend state:')
for label, value in backend_snapshot.items():
    print(f'  {label}: {value}')

pd.DataFrame(EXPERIMENT_DATASET)[['scenario_id', 'category', 'difficulty', 'is_edge_case']].head(12)

Scenario library: 50 total cases
Experiment dataset: 6 cases
Categories: escalation, permissions, review_workflow
Limit applied: 6

Demo backend state:
  billing_account_count: 22
  invoice_count: 14
  billing_reference_topics: 6


,scenario_id,category,difficulty,is_edge_case
0,CR_012,review_workflow,standard,False
1,CR_010,review_workflow,standard,False
2,CR_006,review_workflow,standard,False
3,CP_012,permissions,standard,False
4,CP_005,permissions,standard,False
5,CE_004,escalation,edge,True


In [5]:
PERMISSIONS_CASE = select_cases_by_category(DATASET, 'permissions')[0]
REVIEW_CASE = select_cases_by_category(DATASET, 'review_workflow')[0]
BILLING_CASE = select_cases_by_category(DATASET, 'billing')[0]
ESCALATION_CASE = select_cases_by_category(DATASET, 'escalation')[0]
ROUTING_CATEGORIES = sorted({case['category'] for case in DATASET})

# We are picking a billing case for the demo
ROUTER_DEMO_CASE = BILLING_CASE

---
## Part 1 - `v1` Routing Classification

We start with the simplest system: a router that maps each user message to one support category.

The loop is:
1. Run the router once.
2. Add a code-based exact-match eval.
3. Improve the prompt.
4. Re-run and compare the score.

In [6]:
PROMPT_ROUTER_V1 = (
    'You are a support routing classifier. '
)

router_v1 = run_router_structured(
    client,
    ROUTER_DEMO_CASE['user_input'],
    PROMPT_ROUTER_V1.format(categories=', '.join(ROUTING_CATEGORIES)),
    ROUTING_CATEGORIES,
)
router_payload_v1 = pack_response_payload(router_v1['category'], metadata=router_v1)

display(pd.DataFrame([
    {
        'scenario_id': ROUTER_DEMO_CASE['scenario_id'],
        'user_input': ROUTER_DEMO_CASE['user_input'],
        'expected_category': ROUTER_DEMO_CASE['category'],
        'predicted_category': router_v1['category'],
        **score_routing_response(router_payload_v1, ROUTER_DEMO_CASE['category']),
    }
]))
print(json.dumps(router_v1, indent=2))

,scenario_id,user_input,expected_category,predicted_category,exact_match,exact_match_reasoning,total
0,CB_001,I was billed $49 twice this month. Can you fix that?,billing,escalation,Poor,Predicted `escalation` but expected `billing`.,0.0


{
  "category": "escalation",
  "fallback_reason": "Invalid category returned: I can\u2019t assist with billing issues directly. Please contact customer support for help with your billing inquiry."
}


### Run The Full Dataset In Arize

One demo case is not enough to tell whether the prompt is reliable.
For the full-dataset run, we will use an Arize experiment so the traces and scores live in one place.
We start by running the baseline prompt across the dataset.


In [7]:
router_dataset_bundle = ensure_arize_dataset(
    arize_api_key=ARIZE_API_KEY,
    arize_space_id=ARIZE_SPACE_ID,
    dataset=EXPERIMENT_DATASET,
)
arize_client = router_dataset_bundle['client']
ROUTER_DATASET_ID = router_dataset_bundle['dataset_id']

def task_router_v1(row: dict) -> str:
    route = run_router_structured(
        client,
        row['user_input'],
        PROMPT_ROUTER_V1.format(categories=', '.join(ROUTING_CATEGORIES)),
        ROUTING_CATEGORIES,
    )
    return pack_response_payload(route['category'], metadata=route)

print('Running router baseline experiment without evaluators...')
run_router_v1_traces = run_experiment(
    arize_client=arize_client,
    dataset_id=ROUTER_DATASET_ID,
    name_prefix='router-v1-traces-only',
    task=task_router_v1,
    evaluators=[],
)

print(f"Trace-only router experiment ID: {run_router_v1_traces['experiment'].id}")
print(f"Rows evaluated: {len(run_router_v1_traces['results_df'])}")
display(run_router_v1_traces['results_df'].head(10))
print('\n\nWe now have traces, but no score yet.')


  arize.pre_releases | WARNING | [BETA] datasets.list is a beta API in Arize SDK v8.11.0 and may change without notice. If you experience unexpected failures, please upgrade to the most recent version of the package.
  arize.pre_releases | WARNING | [BETA] datasets.create is a beta API in Arize SDK v8.11.0 and may change without notice. If you experience unexpected failures, please upgrade to the most recent version of the package.
Running router baseline experiment without evaluators...
  arize.experiments.functions | INFO | 🧪 Experiment started.
  arize.experiments.evaluators.executors | WARNING | 🐌!! If running inside a notebook, patching the event loop with nest_asyncio will allow asynchronous eval submission, and is significantly faster. To patch the event loop, run `nest_asyncio.apply()`.


I0407 17:20:13.598533 16156100 ev_poll_posix.cc:593] FD from fork parent still in poll list: fd(98, generation: 1)


running tasks |          | 0/6 (0.0%) | ⏳ 00:00<? | ?it/s

  arize.experiments.functions | INFO | ✅ Task runs completed.
Tasks Summary (04/07/26 05:20 PM -0500)
---------------------------------------
   n_examples  n_runs  n_errors
0           6       6         0
  arize.pre_releases | WARNING | [BETA] experiments.get is a beta API in Arize SDK v8.11.0 and may change without notice. If you experience unexpected failures, please upgrade to the most recent version of the package.
Trace-only router experiment ID: RXhwZXJpbWVudDo3NDk4MTordGdR
Rows evaluated: 6


,id,example_id,result,result.trace.id,result.trace.timestamp
0,EXP_ID_51ffc0,0e1d1e3c-79cd-4058-822c-c2a433c2907a,"{""response_text"": ""escalation"", ""tool_calls"": [], ""action_calls"": [], ""metadata"": {""category"": ""escalation"", ""fallback_reason"": ""Invalid category returned: It sounds like you might be looking for guidance on whether a decision or action requires CEO approval. Could you provide more context or details about the situation? This will help me assist you better.""}}",2d5b460a08316f9df5f0cdc1c56f19ae,1775600413610
1,EXP_ID_f1b5b6,d182654c-2b82-40a8-9247-29a5a7e22a9f,"{""response_text"": ""escalation"", ""tool_calls"": [], ""action_calls"": [], ""metadata"": {""category"": ""escalation"", ""fallback_reason"": ""Invalid category returned: Could you please provide more context or specify which file you are referring to? This will help me assist you better.""}}",c5e9657090495a9edbb8e63a008055d4,1775600414864
2,EXP_ID_f61382,f2fff1fa-f14b-41dd-abc6-5a772f76cf2a,"{""response_text"": ""escalation"", ""tool_calls"": [], ""action_calls"": [], ""metadata"": {""category"": ""escalation"", ""fallback_reason"": ""Invalid category returned: To determine whether a file is waiting on you or someone else, you can follow these steps:\n\n1. **Check the File Status**: Look for any status indicators or comments associated with the file. Many project management tools or file-sharing platforms have features that show who last edited the file or if any actions are pending.\n\n2. **Review Comments or Annotations**: If the file has comments or""}}",bfcb4fe4071b738481c71c99d5ea93c6,1775600416111
3,EXP_ID_c42b94,f991747d-f99c-4e10-b25d-cb377c7d6001,"{""response_text"": ""escalation"", ""tool_calls"": [], ""action_calls"": [], ""metadata"": {""category"": ""escalation"", ""fallback_reason"": ""Invalid category returned: Yes, accessing files shared by another company's workspace typically involves a specific process. Here are some general steps you might need to follow:\n\n1. **Request Access**: Contact the person or team who shared the file and request access. They may need to grant you permission.\n\n2. **Check Permissions**: Ensure that you have the necessary permissions to access the shared file. This may involve being added to""}}",2035a51b6b668358e283632934e6f87e,1775600418480
4,EXP_ID_416c9b,2669a001-bca6-4319-8fb4-7d3f87696d4c,"{""response_text"": ""escalation"", ""tool_calls"": [], ""action_calls"": [], ""metadata"": {""category"": ""escalation"", ""fallback_reason"": ""Invalid category returned: Access to the monthly close checklist typically depends on the organization's policies regarding sensitive financial information. Contractors may or may not have access based on their role and the level of confidentiality required. It's best to check with your supervisor or the finance department to determine if contractors are permitted to access this document.""}}",e2f0808620b15a27057450934a5bca94,1775600420654
5,EXP_ID_f956ec,d9d2a4a5-4a8c-482e-bf72-4802b9674cb4,"{""response_text"": ""escalation"", ""tool_calls"": [], ""action_calls"": [], ""metadata"": {""category"": ""escalation"", ""fallback_reason"": ""Invalid category returned: It sounds like you need urgent assistance with a potential data breach. Please contact your organization's IT or security team immediately to report the incident. They will be able to guide you on the necessary steps to mitigate any risks and address the situation appropriately.""}}",c0ca975e2190581b6b60949f92cb0ee5,1775600422783




We now have traces, but no score yet.


### Add The Evaluator And Re-Run

We now have traces, but no score yet.
Now we add the exact-match evaluator.
This turns the full-dataset run from trace inspection into a measurable benchmark.


In [8]:
router_eval = [ExactMatchEvaluator(expected_field='category')]

print('Running router baseline experiment with exact-match evaluator...')
run_router_v1 = run_experiment(
    arize_client=arize_client,
    dataset_id=ROUTER_DATASET_ID,
    name_prefix='router-v1-baseline',
    task=task_router_v1,
    evaluators=router_eval,
)

router_v1_summary = summarize_router_experiment_results(run_router_v1['results_df'])
display(router_v1_summary)

Running router baseline experiment with exact-match evaluator...
  arize.experiments.functions | INFO | 🧪 Experiment started.
  arize.experiments.evaluators.executors | WARNING | 🐌!! If running inside a notebook, patching the event loop with nest_asyncio will allow asynchronous eval submission, and is significantly faster. To patch the event loop, run `nest_asyncio.apply()`.


running tasks |          | 0/6 (0.0%) | ⏳ 00:00<? | ?it/s

  arize.experiments.functions | INFO | ✅ Task runs completed.
Tasks Summary (04/07/26 05:20 PM -0500)
---------------------------------------
   n_examples  n_runs  n_errors
0           6       6         0
  arize.experiments.evaluators.executors | WARNING | 🐌!! If running inside a notebook, patching the event loop with nest_asyncio will allow asynchronous eval submission, and is significantly faster. To patch the event loop, run `nest_asyncio.apply()`.


running experiment evaluations |          | 0/6 (0.0%) | ⏳ 00:00<? | ?it/s

  arize.experiments.functions | INFO | ✅ All evaluators completed.


,rows_evaluated,score_column,exact_match_accuracy
0,6,eval.ExactMatchEvaluator.score,0.166667


### Fix The Prompt And Re-Run The Same Experiment

Once we have the baseline score, we improve the router prompt and run the same Arize experiment flow again.
That lets us verify whether the score actually improved.


In [11]:
PROMPT_ROUTER = (
    'You are a precise support routing classifier for an AI support copilot. '
    'Classify each user question into exactly one category from this list: {categories}. '
    'Choose permissions for access, sharing, ownership, or admin-rights requests. '
    'Choose review_workflow for approval status, blockers, sign-off, resubmission, or workflow-state questions. '
    'Choose billing for invoices, charges, credits, taxes, pricing, payment methods, renewals, or refunds. '
    'Choose escalation for urgent, high-risk, angry, security-sensitive, legal-risk, or explicitly human-handoff requests. '
    'Return strict JSON with one key: category.'
)

def task_router_v2(row: dict) -> str:
    route = run_router_structured(
        client,
        row['user_input'],
        PROMPT_ROUTER.format(categories=', '.join(ROUTING_CATEGORIES)),
        ROUTING_CATEGORIES,
    )
    return pack_response_payload(route['category'], metadata=route)

print('Running router improved experiment with exact-match evaluator...')
run_router_v2 = run_experiment(
    arize_client=arize_client,
    dataset_id=ROUTER_DATASET_ID,
    name_prefix='router-v2-improved',
    task=task_router_v2,
    evaluators=router_eval,
)

router_v2_summary = summarize_router_experiment_results(run_router_v2['results_df'])

router_comparison = pd.DataFrame([
    {
        'variant': 'router_v1',
        'exact_match_accuracy': router_v1_summary.loc[0, 'exact_match_accuracy'],
        'rows_evaluated': router_v1_summary.loc[0, 'rows_evaluated'],
    },
    {
        'variant': 'router_v2_prompt',
        'exact_match_accuracy': router_v2_summary.loc[0, 'exact_match_accuracy'],
        'rows_evaluated': router_v2_summary.loc[0, 'rows_evaluated'],
    },
])

display(router_comparison)

print(f"Router baseline experiment ID: {run_router_v1['experiment'].id}")
print(f"Router improved experiment ID: {run_router_v2['experiment'].id}")


Running router improved experiment with exact-match evaluator...
  arize.experiments.functions | INFO | 🧪 Experiment started.
  arize.experiments.evaluators.executors | WARNING | 🐌!! If running inside a notebook, patching the event loop with nest_asyncio will allow asynchronous eval submission, and is significantly faster. To patch the event loop, run `nest_asyncio.apply()`.


running tasks |          | 0/6 (0.0%) | ⏳ 00:00<? | ?it/s

  arize.experiments.functions | INFO | ✅ Task runs completed.
Tasks Summary (04/07/26 05:10 PM -0500)
---------------------------------------
   n_examples  n_runs  n_errors
0           6       6         0
  arize.experiments.evaluators.executors | WARNING | 🐌!! If running inside a notebook, patching the event loop with nest_asyncio will allow asynchronous eval submission, and is significantly faster. To patch the event loop, run `nest_asyncio.apply()`.


running experiment evaluations |          | 0/6 (0.0%) | ⏳ 00:00<? | ?it/s

  arize.experiments.functions | INFO | ✅ All evaluators completed.


AttributeError: 'tuple' object has no attribute 'loc'

In [ ]:
print('Open the two router experiments in Arize to compare traces, score distributions, and failure patterns.')

---
## Part 2 - `v2` Routed Draft Replies

The second system is pre-scaffolded.
Participants do not build the specialists from scratch.
They run the routed system first, then improve prompts and evaluate reliability.

The specialist implementations are intentionally different:
- `permissions`: prompt-only
- `review_workflow`: prompt plus injected context
- `billing`: tool-using ReAct-style agent
- `escalation`: human handoff response

In [ ]:
PROMPT_PERMISSIONS = (
    'You are a permissions support specialist. '
    'Answer access questions clearly and concisely. '
    'Explain the right request path, who can grant access, and any policy limitation. '
    'Do not claim you changed permissions or approved access.'
)

PROMPT_REVIEW_WORKFLOW = (
    'You are a review-workflow support specialist. '
    'Use the provided workflow context to explain the current stage, blockers, and next best step. '
    'Do not invent missing approvers or workflow states.'
)

PROMPT_BILLING = (
    'You are a billing support specialist for an AI support copilot. '
    'The authenticated billing account ID for this session is: {authenticated_account_id}. '
    'Use get_billing_account and get_invoice_details to verify facts before replying. '
    'Use read_billing_reference for policy guidance. '
    'If a billing credit or human handoff is required, take the action before replying. '
    'Use apply_billing_credit or escalate_to_human when appropriate.'
)

ESCALATION_RESPONSE_TEMPLATE = (
    "I'm sorry this requires urgent human attention. "
    "I've escalated the case for {account_name} to a human specialist, and they will follow up directly."
)


In [ ]:
print('Billing tools available to the routed system:')
for tool in TOOLS:
    print(f'  - {tool.name}')

In [ ]:
def run_v2_routed_demo(case: dict):
    route = run_router_structured(
        client,
        case['user_input'],
        PROMPT_ROUTER.format(categories=', '.join(ROUTING_CATEGORIES)),
        ROUTING_CATEGORIES,
    )
    payload = dispatch_specialist_response(
        client=client,
        route_category=route['category'],
        case=case,
        prompt_permissions=PROMPT_PERMISSIONS,
        prompt_review_workflow=PROMPT_REVIEW_WORKFLOW,
        prompt_billing=PROMPT_BILLING,
        escalation_response_template=ESCALATION_RESPONSE_TEMPLATE,
    )
    return route, json.loads(payload)

def summarize_v2_case(case: dict) -> dict:
    route, payload = run_v2_routed_demo(case)
    return {
        'scenario_id': case['scenario_id'],
        'expected_category': case['category'],
        'predicted_category': route['category'],
        'response_text': payload['response_text'],
        'tool_calls': payload.get('tool_calls', []),
        'action_calls': payload.get('action_calls', []),
    }

In [ ]:
specialist_demo = pd.DataFrame([
    summarize_v2_case(PERMISSIONS_CASE),
    summarize_v2_case(REVIEW_CASE),
    summarize_v2_case(BILLING_CASE),
    summarize_v2_case(ESCALATION_CASE),
])

display(specialist_demo[['scenario_id', 'expected_category', 'predicted_category', 'response_text']])
display(specialist_demo[['scenario_id', 'tool_calls', 'action_calls']])

### Code Eval For `escalate_to_human`

For `v2`, the main code-based metric is whether the system escalates when it should.
This stays intentionally binary so the failure mode is obvious.

In [ ]:
escalation_eval_rows = []
for case in EXPERIMENT_DATASET:
    route, payload = run_v2_routed_demo(case)
    predicted_escalate = (
        payload.get('metadata', {}).get('route_category') == 'escalation'
        or any(call.get('name') == 'escalate_to_human' for call in payload.get('action_calls', []))
    )
    label, reasoning = escalation_decision_result(case, json.dumps(payload))
    escalation_eval_rows.append(
        {
            'scenario_id': case['scenario_id'],
            'category': case['category'],
            'expected_escalate': case.get('action_type') == 'escalate_to_human',
            'predicted_escalate': predicted_escalate,
            'label': label,
            'reasoning': reasoning,
        }
    )

escalation_eval_df = pd.DataFrame(escalation_eval_rows)
print(f"Escalate-to-human exact-match accuracy: {(escalation_eval_df['label'] == 'Good').mean():.2%}")
display(escalation_eval_df.head(12))
display(escalation_eval_df[escalation_eval_df['label'] != 'Good'].head(10))


### Trace Annotation Export

The annotation demo happens outside the notebook, but this cell produces a small spreadsheet-ready sample.
It is useful for walking through 10-20 examples, grouping failures, and calibrating LLM judges against human preferences.

In [ ]:
annotation_sample = []
for category in ROUTING_CATEGORIES:
    annotation_sample.extend(select_cases_by_category(EXPERIMENT_DATASET, category)[:3])

annotation_rows = []
for case in annotation_sample:
    route, payload = run_v2_routed_demo(case)
    annotation_rows.append(
        {
            'scenario_id': case['scenario_id'],
            'expected_category': case['category'],
            'routed_category': route['category'],
            'user_input': case['user_input'],
            'expected_output': case['expected_output'],
            'response_text': payload['response_text'],
            'tool_calls_json': json.dumps(payload.get('tool_calls', [])),
            'action_calls_json': json.dumps(payload.get('action_calls', [])),
            'brand_voice_label': '',
            'helpfulness_label': '',
            'annotator_notes': '',
        }
    )

annotation_df = pd.DataFrame(annotation_rows)
annotation_path = REPO_ROOT / 'trace_annotation_sample.xlsx'
annotation_df.to_excel(annotation_path, index=False)

display(annotation_df)
print(f'Wrote annotation sample to: {annotation_path}')

---
## Part 3 - LLM Judges For Reply Quality

After the binary escalation check, we add richer quality signals.
For this session, the judge metrics are:
- `brand_voice`
- `helpfulness`

The prompt-calibration workflow is:
1. inspect traces
2. compare to human-desired responses
3. refine the judge prompt
4. re-run the experiment

In [ ]:
JUDGE_PROMPTS = {
    'brand_voice': (
        'You evaluate AI support responses for brand voice.\n\n'
        'Return exactly two lines in this format:\n'
        'LABEL: <Good|Acceptable|Poor>\n'
        'REASONING: <one sentence>\n\n'
        'GOOD: Calm, professional, concise, and empathetic without sounding robotic.\n'
        'ACCEPTABLE: Mostly professional but generic, awkward, or slightly flat.\n'
        'POOR: Blaming, abrupt, defensive, or off-brand.'
    ),
    'helpfulness_system': 'You are a strict evaluator of support response helpfulness.',
    'helpfulness': (
        'Evaluate whether the response is helpful for the user question.\n\n'
        'User question:\n{user_input}\n\n'
        'Reference answer:\n{ideal}\n\n'
        'Actual response:\n{actual}\n\n'
        'Return exactly two lines in this format:\n'
        'LABEL: <Good|Acceptable|Poor>\n'
        'REASONING: <one sentence>\n\n'
        'GOOD: Gives the right next step or resolution with the key details.\n'
        'ACCEPTABLE: Directionally right but incomplete or vague.\n'
        'POOR: Misleading, unhelpful, or misses the main user need.'
    ),
}


In [ ]:
route_v2, payload_v2 = run_v2_routed_demo(BILLING_CASE)
payload_v2_json = json.dumps(payload_v2)

brand_voice_label, brand_voice_reasoning = judge_brand_voice(client, payload_v2_json, JUDGE_PROMPTS)
helpfulness_label, helpfulness_reasoning = judge_helpfulness(client, payload_v2_json, BILLING_CASE, JUDGE_PROMPTS)
escalation_label, escalation_reasoning = escalation_decision_result(BILLING_CASE, payload_v2_json)

scorecard_v2 = pd.DataFrame([
    {
        'scenario_id': BILLING_CASE['scenario_id'],
        'route_category': route_v2['category'],
        'brand_voice': brand_voice_label,
        'helpfulness': helpfulness_label,
        'escalate_to_human': escalation_label,
    }
])

reasoning_v2 = pd.DataFrame([
    {
        'brand_voice_reasoning': brand_voice_reasoning,
        'helpfulness_reasoning': helpfulness_reasoning,
        'escalation_reasoning': escalation_reasoning,
    }
])

display(scorecard_v2)
display(reasoning_v2)
print(payload_v2['response_text'])

### Error Analysis Prompts

Useful questions for the session:
- Which escalation misses are routing errors versus specialist behavior errors?
- Which billing traces used the wrong tools or skipped the required action?
- Which judge disagreements mean the judge prompt still needs calibration?

---
## Arize Experiments For `v2`

The routed copilot experiment uses:
- `brand_voice` judge
- `helpfulness` judge
- `escalate_to_human` code eval

In [ ]:
experiment_bundle = prepare_experiment_bundle(
    client=client,
    arize_api_key=ARIZE_API_KEY,
    arize_space_id=ARIZE_SPACE_ID,
    dataset=DATASET,
    prompt_router=PROMPT_ROUTER,
    prompt_permissions=PROMPT_PERMISSIONS,
    prompt_review_workflow=PROMPT_REVIEW_WORKFLOW,
    prompt_billing=PROMPT_BILLING,
    escalation_response_template=ESCALATION_RESPONSE_TEMPLATE,
    judge_prompts=JUDGE_PROMPTS,
    limit_n=LIMIT_N_CASES,
)
arize_client = experiment_bundle['client']
DATASET_ID = experiment_bundle['dataset_id']
tasks = experiment_bundle['tasks']

In [ ]:
print('Running routed copilot experiment...')
run_v2_routed = run_experiment(
    arize_client=arize_client,
    dataset_id=DATASET_ID,
    name_prefix='v2-routed-copilot',
    task=tasks['task_v2_routed'],
    evaluators=experiment_bundle['build_evaluators']('v2_routed'),
)

print(f"Routed copilot experiment ID: {run_v2_routed['experiment'].id}")
print(f"Rows evaluated: {len(run_v2_routed['results_df'])}")

## Online Evaluation Note

Offline evals are not enough by themselves.
The next production step is to add online signals in Arize, for example:
- route override rate by humans
- escalation rate by category
- unresolved follow-up rate
- CSAT or helpfulness trend by category

That concept is in scope for discussion, but not a full notebook walkthrough.

---
## Production Readiness

Use the route experiment, the escalation code eval, the trace sample, and the LLM-judge results to fill in this checklist.

In [ ]:
for row in format_checklist_rows(production_readiness_checklist()):
    print(row)